# 🏛️ RAG Architecture Overview

**Day 7 — Notebook 1 of 4 | Estimated Time: 25 minutes**

---

## What You'll Learn
- What RAG is and the two problems it solves
- The Retriever + Generator pattern explained
- When to use RAG vs. fine-tuning vs. prompt-only approaches
- The anatomy of a RAG query from start to finish

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
MODEL_ID = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. The Two Problems RAG Solves

LLMs are powerful but have two fundamental limitations:

### Problem 1: Knowledge Cutoff
LLMs are trained on data up to a certain date. They know nothing about:
- Events after their training cutoff
- Your private documents (internal wikis, emails, contracts)
- Real-time data (stock prices, live sports scores, current weather)

### Problem 2: Hallucination
When asked about something they don't know, LLMs don't say "I don't know" — they generate a plausible-sounding but potentially false answer.

### The RAG Solution
**Retrieval Augmented Generation (RAG)** solves both by:
1. **Retrieving** relevant passages from a knowledge base at query time
2. **Augmenting** the LLM prompt with those passages
3. **Generating** an answer grounded in the retrieved evidence

```
Without RAG:  [Question] → LLM → Answer (may hallucinate)

With RAG:     [Question] → Retriever → [Relevant Docs]
                                              ↓
              [Question + Docs] → LLM → Grounded Answer
```

---

## 2. The Retriever + Generator Pattern

RAG has two components that work together:

```
╔══════════════════════════════════════════════════════════════╗
║                     INDEXING (done once)                     ║
║                                                              ║
║  Documents → Chunk → Embed → Store in Vector DB              ║
╚══════════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════════╗
║                  RETRIEVAL + GENERATION (per query)          ║
║                                                              ║
║  User Query                                                  ║
║       │                                                      ║
║       ▼                                                      ║
║  Embed Query ──────► Vector DB ──► Top-K Chunks              ║
║                                        │                     ║
║                                        ▼                     ║
║  [Query + Context] ──────────────► Gemini LLM                ║
║                                        │                     ║
║                                        ▼                     ║
║                                   Final Answer               ║
╚══════════════════════════════════════════════════════════════╝
```

### The Retriever
- Takes the user's query
- Embeds it using the same embedding model used at indexing time
- Finds the most semantically similar chunks in the vector database
- Returns the top-K most relevant passages

### The Generator
- Receives the user's query **plus** the retrieved passages as context
- Generates a response that is **grounded** in the retrieved evidence
- Can be instructed to cite sources and refuse to answer if evidence is lacking

---

## 3. RAG vs. Alternatives: Decision Framework

RAG is not always the right tool. Here's when to use what:

| Approach | Best When | Drawbacks |
|----------|-----------|----------|
| **Prompt only** | Knowledge is in training data, short context | Hallucinations, knowledge cutoff |
| **RAG** | Private/dynamic/domain-specific knowledge | Retrieval can fail, latency overhead |
| **Fine-tuning** | Specialised style, format, or behaviour | Expensive, doesn't add new facts reliably |
| **Long context** | Small corpus that fits in context window | Expensive, LLM may miss info in long context |
| **RAG + Fine-tuning** | Best of both worlds for production | Most complex and expensive |

**Rule of thumb:**
- Need specific facts from documents? → **RAG**
- Need a different style or format? → **Fine-tuning**
- Just need general intelligence? → **Prompt engineering**

In [ ]:
# Demonstrate the knowledge cutoff problem
response = client.models.generate_content(
    model=MODEL_ID,
    contents="What was the outcome of the 2026 FIFA World Cup final?",
)
print("Without RAG (knowledge cutoff):")
print(response.text)

---

## 4. Anatomy of a RAG Query

Let's trace exactly what happens when a user asks a question in a RAG system.

In [ ]:
# Step-by-step RAG trace (simplified)

# Imagine these are chunks from a company's product documentation
knowledge_base = [
    "ProductX v3.0 was released on 15 January 2025. New features include auto-scaling, "
    "dark mode, and a redesigned dashboard. The minimum system requirement is 8GB RAM.",
    
    "ProductX pricing: Starter plan is free (up to 5 users). Professional plan is $29/month "
    "per seat. Enterprise pricing is negotiated annually. All plans include 99.9% uptime SLA.",
    
    "To install ProductX, download the installer from productx.io/download. Run with admin "
    "privileges. Installation takes approximately 10 minutes. A system restart is required.",
    
    "ProductX integrates with Slack, Jira, GitHub, and Salesforce out of the box. "
    "Custom integrations are available via the REST API. Webhooks support real-time events.",
]

user_question = "How much does ProductX cost for a team of 10 people?"

print("=" * 60)
print("RAG QUERY TRACE")
print("=" * 60)
print(f"\n1. USER QUERY:")
print(f"   '{user_question}'")

print(f"\n2. RETRIEVAL:")
print(f"   → Embed query using text-embedding-004")
print(f"   → Search vector database for top-3 similar chunks")
print(f"   → Retrieved: 'ProductX pricing: Starter plan is free...'")

print(f"\n3. AUGMENTATION:")
print(f"   → Build prompt: [System instruction] + [Retrieved chunks] + [User question]")

print(f"\n4. GENERATION:")
print(f"   → Gemini generates answer grounded in pricing chunk")
print()

# Simulate the generation step
retrieved_context = knowledge_base[1]  # The pricing chunk

response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""Answer using ONLY the context below. Cite the source.

<context>
{retrieved_context}
</context>

Question: {user_question}""",
    config=types.GenerateContentConfig(temperature=0.1),
)
print("5. ANSWER:")
print(f"   {response.text.strip()}")

---

## 5. The Prompt Template

The RAG prompt template is what connects retrieved evidence to the LLM. A good template:
1. Tells the model to use only the context
2. Instructs it to cite sources
3. Tells it what to do when the answer isn't in the context

In [ ]:
RAG_PROMPT_TEMPLATE = """\
You are a helpful assistant that answers questions based on the provided context.

Instructions:
- Answer using ONLY the information in the context below
- If the answer is not in the context, respond: "I don't have enough information to answer this."
- Keep your answer concise and factual
- Reference the source when making specific claims

<context>
{context}
</context>

Question: {question}

Answer:"""

# Test the template
context = "\n\n".join([f"[Doc {i+1}] {doc}" for i, doc in enumerate(knowledge_base[:2])])

prompt = RAG_PROMPT_TEMPLATE.format(
    context=context,
    question="What integrations does ProductX support?",
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(temperature=0.1),
)
print(response.text)

In [ ]:
# Test: question NOT in context — model should refuse gracefully
prompt_out_of_scope = RAG_PROMPT_TEMPLATE.format(
    context=context,
    question="What is the CEO's email address?",
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt_out_of_scope,
    config=types.GenerateContentConfig(temperature=0.1),
)
print("Out-of-scope question:")
print(response.text)

---

## 6. Failure Modes in RAG

RAG can fail in several ways. Understanding them helps you build better systems:

| Failure Mode | Cause | Fix |
|--------------|-------|-----|
| **Retrieval miss** | Relevant chunk not retrieved | Better chunking, more chunks (higher K), query rewriting |
| **Context window overflow** | Too many chunks sent to LLM | Limit K, use compression, summarise chunks |
| **Generator ignores context** | Model uses parametric memory instead | Stronger system instructions, lower temperature |
| **Chunk boundary problem** | Answer spans two chunks | Increase overlap, larger chunks |
| **Stale index** | Knowledge base not updated | Incremental indexing pipeline |
| **Irrelevant retrieval** | Similar-sounding but wrong chunks | Re-ranking, metadata filtering |

---

## 🏋️ Exercise 1: Identify the Failure Mode

Each scenario below represents a RAG failure. Identify which failure mode it is and suggest a fix.

In [ ]:
# Exercise 1: Diagnose RAG failures

scenarios = [
    {
        "description": "User asks 'When was ProductX v3.0 released?' The system retrieves the pricing "
                       "chunk and the integrations chunk, but not the release date chunk.",
        "failure_mode": "???",  # TODO: Fill in
        "fix": "???",           # TODO: Fill in
    },
    {
        "description": "User asks about pricing. The correct pricing chunk is retrieved, but the LLM "
                       "ignores it and answers with outdated pricing it 'remembered' from training data.",
        "failure_mode": "???",
        "fix": "???",
    },
    {
        "description": "The installation guide spans two chunks (chunk 5 ends mid-sentence, chunk 6 "
                       "continues). Only chunk 5 is retrieved, giving an incomplete installation guide.",
        "failure_mode": "???",
        "fix": "???",
    },
]

for i, s in enumerate(scenarios, 1):
    print(f"Scenario {i}: {s['description'][:80]}...")
    print(f"  Failure mode: {s['failure_mode']}")
    print(f"  Fix: {s['fix']}")
    print()

---

## 🏋️ Exercise 2: Design a RAG System

Design a RAG system for a given use case by answering the key design questions.

In [ ]:
# Exercise 2: RAG system design
# Use case: A customer support chatbot for a software product
# The knowledge base contains: user manual (200 pages), FAQ (50 Q&As), release notes (10 versions)

design_questions = {
    "What should the chunk size be?": "???",
    "What overlap is appropriate?": "???",
    "How many chunks (K) should be retrieved per query?": "???",
    "What metadata should be stored with each chunk?": "???",
    "Should any metadata filters be applied before search?": "???",
    "What should the system instruction say?": "???",
    "How should the system handle out-of-scope questions?": "???",
}

print("RAG System Design for Customer Support Chatbot:\n")
for q, a in design_questions.items():
    print(f"Q: {q}")
    print(f"A: {a}")
    print()

---

## Key Takeaways

| Concept | Summary |
|---------|----------|
| **RAG** | Retrieval Augmented Generation — ground LLM answers in retrieved evidence |
| **Retriever** | Embeds the query and finds the most relevant chunks in a vector database |
| **Generator** | LLM that produces answers conditioned on retrieved context |
| **RAG prompt** | Template that injects retrieved chunks between the system instruction and the question |
| **When to use RAG** | Private docs, dynamic data, knowledge cutoff, factual grounding requirements |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| RAG Paper (Lewis et al.) | Paper | [arxiv.org/abs/2005.11401](https://arxiv.org/abs/2005.11401) |
| RAG Explained — IBM | Video | [youtube.com/watch?v=T-D1OfcDW1M](https://www.youtube.com/watch?v=T-D1OfcDW1M) |

---

**Next up:** [02_Building_a_Basic_RAG_System.ipynb](./02_Building_a_Basic_RAG_System.ipynb)